# Pump It Up baseline modelling frame

Build the modelling-data handoff, local evaluation design and majority-class reference for the formal baseline workflow. This notebook deliberately stops before learned preprocessing or feature-based model fitting.

The notebook keeps the workflow visible while delegating mechanical validation, preparation and partitioning to `src/modelling_data.py` and `src/data_partitioning.py`:

1. load the immutable competition source files;
2. prepare the labelled original data and separate competition-scoring data;
3. reserve an untouched local test set;
4. freeze stratified cross-validation folds across the remaining development rows;
5. inspect the resulting membership, balance and fingerprints; and
6. establish the accuracy and per-class recall of an always-majority reference.

In this notebook, **original** means the complete labelled data before the local split. **Competition** means the unlabelled `TestSetValues.csv` rows that DrivenData scores; those rows are never used for local model selection.

The partition below completes course Step 7, **Partition the data**.

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks' or NOTEBOOK_DIR.parent.name != 'stage-1-pump-it-up':
    raise RuntimeError(
        'Run this notebook from the stage-1-pump-it-up/notebooks directory.'
    )

STAGE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from baseline_evaluation import evaluate_majority_reference
from data_partitioning import (
    make_cross_validation,
    partition_modelling_data,
    summarise_partitioned_data,
)
from modelling_data import prepare_modelling_data, summarise_modelling_data

## Load immutable source frames

No cleaned CSV is created. Every run reconstructs the modelling frame from the competition downloads recorded in `data/README.md`.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')

## Prepare modelling data

Validate the source contracts, apply the three settled structural removals, retain IDs as metadata and align `status_group` by identifier. The competition frame receives the same fixed structural policy but remains outside local model selection.

In [3]:
modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)

## Inspect the handoff

The preparation helper has already enforced the modelling-data contract. These tables expose the handoff that the partitioning and cross-validation work will consume.

In [4]:
checkpoint, target_summary = summarise_modelling_data(modelling_data)

display(checkpoint)
display(target_summary.style.format({'share': '{:.2%}'}))

,rows,columns,role
object,,,
X_original,59400,36,Predictors before local split
y_original,59400,1,Target before local split
original_ids,59400,1,IDs before local split
X_competition,14850,36,Unlabelled competition predictors
competition_ids,14850,1,Competition submission IDs


,rows,share
status_group,,
functional,32259,54.31%
functional needs repair,4317,7.27%
non functional,22824,38.42%


## Freeze the local evaluation design

Reserve 20% of the original rows as an untouched stratified local test set. Use five stratified folds across the remaining development rows, so every development row serves as validation data exactly once. Both deterministic operations use seed `20260820`, recording when this design was frozen.

The competition data remains outside the local test partition and every cross-validation fold.

In [5]:
partitioned_data = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned_data)

## Inspect partitions and folds

The fingerprints record exact membership without exposing identifiers. Class shares confirm that the local test set and each rotating validation fold preserve the imbalanced three-class target structure.

In [6]:
partition_settings, partition_summary, fold_summary = summarise_partitioned_data(
    partitioned_data
)
percentage_format = {
    column: '{:.2%}' for column in partition_summary.columns if column != 'rows'
}

display(partition_settings)
display(partition_summary.style.format(percentage_format))
display(fold_summary.style.format(percentage_format))

,value
setting,
Local test fraction,0.2
Local test seed,20260820
Cross-validation folds,5
Cross-validation seed,20260820
Development ID fingerprint,c8a9e27264e4932140ffe9f19c897229230f9da00126f7...
Local test ID fingerprint,6b5aff2b3b7d8dc3bc6ee1b36defabcfef312b03fef940...
Cross-validation fingerprint,bd7da743e9b4498888fa9a7b4166f64266a5f86c5175b7...


,rows,share_of_parent,functional,functional needs repair,non functional
partition,,,,,
Development,47520,80.00%,54.31%,7.27%,38.42%
Local test,11880,20.00%,54.31%,7.26%,38.43%


,rows,share_of_parent,functional,functional needs repair,non functional
validation_fold,,,,,
Fold 1,9504,20.00%,54.31%,7.26%,38.43%
Fold 2,9504,20.00%,54.31%,7.27%,38.42%
Fold 3,9504,20.00%,54.30%,7.27%,38.43%
Fold 4,9504,20.00%,54.30%,7.27%,38.43%
Fold 5,9504,20.00%,54.30%,7.27%,38.43%


## Establish the majority-class reference

Fit `DummyClassifier(strategy='most_frequent')` independently in each development training fold and score it on the corresponding validation fold. This reference measures what accuracy is available from always predicting the locally most common class, while its per-class recall exposes the cost of doing so. The local test and competition data remain unused.

In [7]:
baseline_fold_results, baseline_summary = evaluate_majority_reference(
    partitioned_data,
    cross_validation,
)

display(baseline_fold_results.style.format('{:.2%}'))
display(baseline_summary.style.format('{:.2%}'))

,accuracy,recall: functional,recall: functional needs repair,recall: non functional
validation_fold,,,,
1,54.31%,100.00%,0.00%,0.00%
2,54.31%,100.00%,0.00%,0.00%
3,54.30%,100.00%,0.00%,0.00%
4,54.30%,100.00%,0.00%,0.00%
5,54.30%,100.00%,0.00%,0.00%


,mean,std,min,max
metric,,,,
accuracy,54.31%,0.01%,54.30%,54.31%
recall: functional,100.00%,0.00%,100.00%,100.00%
recall: functional needs repair,0.00%,0.00%,0.00%,0.00%
recall: non functional,0.00%,0.00%,0.00%,0.00%


### Interpretation

The majority reference averages **54.31% validation accuracy** by predicting `functional` for every row. It therefore achieves **100% recall** for `functional` and **0% recall** for both `functional needs repair` and `non functional`. A useful feature-based model must improve on this accuracy while recovering meaningful recall for the two classes the reference entirely misses.

## Next step

Create `03-model-comparison.ipynb`. Define fold-fitted preprocessing there, then compare the first feature-based candidates with this majority reference across the same frozen development folds. Keep the local test set untouched until model selection is complete, and keep `modelling_data.X_competition` outside all local evaluation.